<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M12/M12_Workshop1_Claude_Agent_SDK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🔮 M12 Workshop 1 — Claude Agent SDK</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐⭐ Difficulty: Advanced &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Install and configure the <b>Anthropic Python SDK</b></li>
    <li>Use Claude's <b>tool_use</b> feature to give the model tools</li>
    <li>Build an <b>agent</b> that reasons and calls tools autonomously</li>
    <li>Compare building agents with <b>raw API</b> vs <b>SDK patterns</b></li>
    <li>Understand Claude's approach to <b>structured tool execution</b></li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Add `ANTHROPIC_API_KEY` to Colab Secrets (key icon → Add a secret → Name: `ANTHROPIC_API_KEY`).
>
> Also keep `OPENAI_API_KEY` for comparison.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade anthropic openai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

# OpenAI (for comparison)
openai_client = setup_openai()

# Anthropic
import anthropic
from google.colab import userdata
import json

claude_client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print(f"✅ Anthropic SDK version: {anthropic.__version__}")
print("✅ Both OpenAI and Anthropic clients ready")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Claude API Basics</h2>
</div>

Let's start with a simple Claude API call to see how it compares to OpenAI.

In [ ]:
# ============================================================
# 🤖 Basic Claude API Call
# ============================================================

# Claude API call
response = claude_client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{"role": "user", "content": "What is LangChain in one sentence?"}]
)

print(f"🟣 Claude: {response.content[0].text}")
print(f"📊 Tokens: input={response.usage.input_tokens}, output={response.usage.output_tokens}")
print(f"🔚 Stop reason: {response.stop_reason}")

# Compare with OpenAI
oai_resp = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What is LangChain in one sentence?"}]
)
print(f"\n🔵 GPT:    {oai_resp.choices[0].message.content}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Key API differences: OpenAI uses <code>client.chat.completions.create()</code> with <code>choices[0].message.content</code>. Anthropic uses <code>client.messages.create()</code> with <code>content[0].text</code>. The message format is similar but not identical.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Claude Tool Use</h2>
</div>

Claude's tool use ("function calling") works differently from OpenAI. Tools are defined with the same JSON Schema, but the response format and execution loop differ.

In [ ]:
# ============================================================
# 🔧 Define Tools for Claude
# ============================================================

# Tool implementations
def calculator(expression: str) -> str:
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expression):
        return "Error: Invalid characters"
    return str(round(eval(expression), 4))

def web_search(query: str) -> str:
    knowledge = {
        "population of boston": "Boston has a population of approximately 675,647 (2024).",
        "universities in boston": "Boston has approximately 35 colleges and universities.",
        "gdp of usa": "The US GDP in 2024 was approximately $28.78 trillion.",
    }
    for key, val in knowledge.items():
        if key in query.lower():
            return val
    return f"No results for: {query}"

# Claude tool schemas (slightly different format from OpenAI)
claude_tools = [
    {
        "name": "calculator",
        "description": "Evaluate a mathematical expression.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression to evaluate"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for factual information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    }
]

tool_handlers = {
    "calculator": lambda args: calculator(args["expression"]),
    "web_search": lambda args: web_search(args["query"]),
}

print(f"✅ Defined {len(claude_tools)} tools for Claude")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Claude Agent Loop</h2>
</div>

The Claude agent loop is similar to our M06 ReAct agent, but uses Claude's native tool_use format.

In [ ]:
# ============================================================
# 🔄 Claude Agent Loop
# ============================================================

def claude_agent(question: str, max_steps: int = 5, verbose: bool = True) -> str:
    """An agent powered by Claude with tool use."""
    messages = [{"role": "user", "content": question}]

    if verbose:
        print(f"\n{'='*70}")
        print(f"🧑 Question: {question}")
        print(f"{'='*70}")

    for step in range(1, max_steps + 1):
        response = claude_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=claude_tools,
            messages=messages
        )

        # Check if Claude wants to use tools
        if response.stop_reason == "tool_use":
            # Add Claude's response (may contain text + tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})

            # Process each tool use block
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"\n🟣 Step {step} — Tool Call")
                        print(f"   🛠️  Tool: {block.name}")
                        print(f"   📥 Args: {json.dumps(block.input)[:100]}")

                    # Execute the tool
                    result = tool_handlers.get(block.name, lambda _: "Unknown tool")(block.input)

                    if verbose:
                        print(f"   👁️  Result: {result}")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Feed tool results back
            messages.append({"role": "user", "content": tool_results})

        elif response.stop_reason == "end_turn":
            # Claude is done — extract final text
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            if verbose:
                print(f"\n🟢 Final Answer:")
                print(f"   {final}")
                print(f"{'='*70}")
            return final

    return "Agent exceeded max steps."

print("✅ Claude agent ready!")

In [ ]:
# ============================================================
# 🧪 Test: Multi-Step Question
# ============================================================
claude_agent("What is the population of Boston divided by the number of universities?")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Raw API vs SDK Comparison</h2>
</div>

Let's compare the same agent task implemented with the raw API pattern (from M06) vs the SDK approach.

In [ ]:
# ============================================================
# 🔄 Side-by-Side: OpenAI vs Claude Tool Schemas
# ============================================================

print("📋 Tool Schema Comparison:\n")
print("🔵 OpenAI Format:")
print(json.dumps({
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "...",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}}
    }
}, indent=2))

print("\n🟣 Anthropic Format:")
print(json.dumps({
    "name": "calculator",
    "description": "...",
    "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}}
}, indent=2))

print("\n📋 Key Differences:")
print("  1. OpenAI wraps in 'function' key; Anthropic uses flat structure")
print("  2. OpenAI: 'parameters'; Anthropic: 'input_schema'")
print("  3. OpenAI: tool_calls in message; Anthropic: tool_use content blocks")
print("  4. OpenAI: 'tool' role for results; Anthropic: tool_result in user message")

In [ ]:
# ============================================================
# 🧪 Same Question, Both Models
# ============================================================
question = "What is 15% of the GDP of the USA?"

print("🟣 Claude Agent:")
claude_answer = claude_agent(question)

# OpenAI agent (simplified)
print("\n🔵 GPT Agent:")
oai_tools = [
    {"type": "function", "function": {"name": "calculator", "description": "Evaluate math.", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "web_search", "description": "Search for facts.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
]

msgs = [{"role": "user", "content": question}]
for _ in range(5):
    r = openai_client.chat.completions.create(model="gpt-4.1-mini", messages=msgs, tools=oai_tools, temperature=0)
    m = r.choices[0].message
    if m.tool_calls:
        msgs.append(m)
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            result = tool_handlers[tc.function.name](args)
            print(f"   🛠️ {tc.function.name}({args}) → {result}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    else:
        print(f"   🤖 {m.content}")
        break

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Both Claude and GPT support tool use with the same core pattern: <b>define tools → send to model → execute tool calls → feed results back → repeat</b>. The schema format differs slightly, but the agent architecture is identical. Knowing both SDKs makes you model-agnostic.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "How does Claude indicate it wants to call a tool?",
        "options": [
            "It returns stop_reason='tool_use' with tool_use content blocks",
            "It puts tool calls in a 'function_call' field on the message",
            "It returns a special HTTP 202 status code",
            "It writes the tool name in plain text and you parse it"
        ],
        "answer": 0,
        "explanation": "Claude signals tool use via stop_reason='tool_use' and includes tool_use blocks in the content array. Each block has an id, name, and input. You execute the tool, then send back a tool_result with the matching id."
    },
    {
        "q": "What is the key schema difference between OpenAI and Anthropic tool definitions?",
        "options": [
            "OpenAI uses YAML; Anthropic uses JSON",
            "OpenAI wraps tools in {type: 'function', function: {...}}; Anthropic uses a flat schema with 'input_schema'",
            "OpenAI requires Python type hints; Anthropic requires TypeScript",
            "There is no difference — both use the exact same format"
        ],
        "answer": 1,
        "explanation": "OpenAI nests tool definitions inside {type: 'function', function: {name, description, parameters}}. Anthropic uses a flat structure with {name, description, input_schema}. The JSON Schema for parameters is identical in both."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add a Custom Tool

Add a `file_reader` tool to the Claude agent. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Add a File Reader Tool
# ============================================================
# Replace each ----- with the correct value

MOCK_FILES = {
    "report.txt": "Q4 2024 Revenue: $2.3M. Growth: 15%. Top market: Enterprise.",
    "todo.txt": "1. Review budget 2. Hire engineer 3. Launch feature",
}

def file_reader(filename: str) -> str:
    return MOCK_FILES.get(filename, f"File '{filename}' not found.")

# Add the new tool schema (Anthropic format)
new_tool = {
    "name": "-----",                    # Tool name?
    "description": "Read the contents of a file. Available: report.txt, todo.txt.",
    "-----": {                           # What key for the schema? (hint: not 'parameters')
        "type": "object",
        "properties": {
            "filename": {"type": "string", "description": "Name of file to read"}
        },
        "required": ["-----"]            # Required field name?
    }
}

# Extended tools and handlers
extended_tools = claude_tools + [new_tool]
extended_handlers = {**tool_handlers, "file_reader": lambda args: file_reader(args["filename"])}

# Test with extended tools
messages = [{"role": "user", "content": "What does the report file say about revenue? Also calculate 15% growth on $2.3M."}]
for _ in range(5):
    resp = claude_client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=1024,
        tools=extended_tools, messages=messages
    )
    if resp.stop_reason == "tool_use":
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                res = extended_handlers[block.name](block.input)
                print(f"🛠️ {block.name} → {res}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": res})
        messages.append({"role": "user", "content": results})
    else:
        print(f"\n🤖 Final: {resp.content[0].text}")
        break

**📝 Your Observations:** *(double-click to edit this cell)*

1. How did Claude's tool calling behavior differ from GPT's? _[Your answer]_

2. Which SDK do you find more intuitive? Why? _[Your answer]_

3. How would you build a model-agnostic agent that works with both? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Streaming:</b> Use <code>claude_client.messages.stream()</code> to see Claude's response in real-time</li>
    <li><b>System prompt:</b> Add a system parameter to shape Claude's personality and behavior</li>
    <li><b>Multi-turn:</b> Build a chat loop where Claude remembers previous tool results across turns</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Workshop 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built agents with the Anthropic SDK and compared Claude vs GPT tool use patterns.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li>Claude uses <code style="color: #90caf9;">input_schema</code> (vs OpenAI's <code style="color: #90caf9;">parameters</code>)</li>
      <li>Claude returns <code style="color: #90caf9;">tool_use</code> content blocks (vs OpenAI's <code style="color: #90caf9;">tool_calls</code>)</li>
      <li>The <b>agent loop pattern is identical</b> — only the schema format differs</li>
      <li>Knowing both SDKs makes you <b>model-agnostic</b> — the most valuable skill</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M12 Workshop 2: Google ADK</p>
</div>